# Generate Synthetic Company Data

Creates 4 tables with referential integrity:
- **customers** (~2,500 rows) — tier, region, industry segmentation
- **products** (~500 rows) — log-normal pricing, categories
- **orders** (~25,000 rows) — weekday/weekend patterns, Q4 uptick
- **order_items** (~75,000 rows) — Pareto product distribution

In [ ]:
# Parameters (set by DABs job)
dbutils.widgets.text("catalog", "startups_catalog")
dbutils.widgets.text("schema", "dw_genie")
dbutils.widgets.text("num_customers", "2500")
dbutils.widgets.text("num_products", "500")
dbutils.widgets.text("num_orders", "25000")
dbutils.widgets.text("date_range_days", "180")
dbutils.widgets.text("seed", "42")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
NUM_CUSTOMERS = int(dbutils.widgets.get("num_customers"))
NUM_PRODUCTS = int(dbutils.widgets.get("num_products"))
NUM_ORDERS = int(dbutils.widgets.get("num_orders"))
DATE_RANGE_DAYS = int(dbutils.widgets.get("date_range_days"))
SEED = int(dbutils.widgets.get("seed"))

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
print(f"Config: {CATALOG}.{SCHEMA}, {NUM_CUSTOMERS} customers, {NUM_PRODUCTS} products, {NUM_ORDERS} orders")

In [ ]:
# Set Unity Catalog context and create infrastructure
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw_data")
print(f"Schema and volume ready: {CATALOG}.{SCHEMA}")

In [ ]:
import math
import random
from datetime import datetime, timedelta
from pyspark.sql.types import (
    DateType, DoubleType, IntegerType, LongType, StringType, StructField, StructType
)

rng = random.Random(SEED)

# --- Distributions ---
TIERS = ["Free", "Pro", "Enterprise"]
TIER_WEIGHTS = [0.5, 0.35, 0.15]
REGIONS = ["North America", "Europe", "Asia Pacific", "Latin America", "Middle East & Africa"]
INDUSTRIES = ["Technology", "Healthcare", "Finance", "Manufacturing", "Retail", "Energy", "Education", "Media", "Transportation", "Real Estate", "Agriculture", "Government"]
CATEGORIES = ["Software", "Hardware", "Services", "Consulting", "Maintenance", "Training", "Subscriptions", "Infrastructure"]
ORDER_STATUSES = ["completed", "processing", "shipped", "cancelled", "refunded"]
STATUS_WEIGHTS = [0.65, 0.12, 0.13, 0.07, 0.03]

# Region bounding boxes for realistic lat/lon generation (lat_min, lat_max, lon_min, lon_max)
REGION_BOUNDS = {
    "North America": (25.0, 50.0, -125.0, -70.0),
    "Europe": (36.0, 60.0, -10.0, 30.0),
    "Asia Pacific": (10.0, 45.0, 100.0, 145.0),
    "Latin America": (-35.0, 20.0, -80.0, -35.0),
    "Middle East & Africa": (-10.0, 35.0, 20.0, 55.0),
}

FIRST_NAMES = ["James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael", "Linda", "David", "Elizabeth", "William", "Barbara", "Richard", "Susan", "Joseph", "Jessica", "Thomas", "Sarah", "Christopher", "Karen", "Charles", "Lisa", "Daniel", "Nancy", "Matthew", "Betty", "Anthony", "Margaret", "Mark", "Sandra", "Donald", "Ashley", "Steven", "Kimberly", "Andrew", "Emily", "Paul", "Donna", "Joshua", "Michelle"]
LAST_NAMES = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson"]
COMPANY_SUFFIXES = ["Inc", "LLC", "Corp", "Group", "Solutions", "Systems", "Technologies"]
PRODUCT_ADJECTIVES = ["Advanced", "Pro", "Elite", "Core", "Premium", "Standard", "Ultra", "Flex", "Smart", "Rapid", "Secure", "Cloud"]
PRODUCT_NOUNS = ["Platform", "Suite", "Engine", "Module", "Toolkit", "Gateway", "Hub", "Manager", "Optimizer", "Dashboard", "Connector", "Agent"]

print("Distributions loaded")

In [ ]:
# --- Generate Customers (with lat/lon) ---
rows = []
for i in range(1, NUM_CUSTOMERS + 1):
    tier = rng.choices(TIERS, weights=TIER_WEIGHTS, k=1)[0]
    first, last = rng.choice(FIRST_NAMES), rng.choice(LAST_NAMES)
    suffix = rng.choice(COMPANY_SUFFIXES)
    region = rng.choice(REGIONS)
    lat_min, lat_max, lon_min, lon_max = REGION_BOUNDS[region]
    latitude = round(rng.uniform(lat_min, lat_max), 6)
    longitude = round(rng.uniform(lon_min, lon_max), 6)
    rows.append((
        i, f"{first} {last}",
        f"{first.lower()}.{last.lower()}@{last.lower()}{suffix.lower()}.com",
        f"{first} {last} {suffix}", tier,
        region, rng.choice(INDUSTRIES),
        datetime(2020 + rng.randint(0, 4), rng.randint(1, 12), rng.randint(1, 28)),
        latitude, longitude,
    ))

schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("company", StringType(), True),
    StructField("tier", StringType(), False),
    StructField("region", StringType(), False),
    StructField("industry", StringType(), False),
    StructField("signup_date", DateType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
])
spark.createDataFrame(rows, schema).write.mode("overwrite").parquet(f"{VOLUME_PATH}/customers/")
print(f"Generated {NUM_CUSTOMERS} customers (with lat/lon)")

In [ ]:
# --- Generate Products ---
rows = []
for i in range(1, NUM_PRODUCTS + 1):
    adj, noun = rng.choice(PRODUCT_ADJECTIVES), rng.choice(PRODUCT_NOUNS)
    version = rng.choice(["", " v2", " v3", " Plus", " Lite"])
    unit_price = round(max(5.0, min(math.exp(rng.gauss(3.5, 1.2)), 5000.0)), 2)
    cost = round(unit_price * rng.uniform(0.3, 0.7), 2)
    rows.append((i, f"{adj} {noun}{version}", rng.choice(CATEGORIES), unit_price, cost, rng.random() < 0.75))

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("cost", DoubleType(), False),
    StructField("is_active", StringType(), False),
])
rows = [(r[0], r[1], r[2], r[3], r[4], str(r[5])) for r in rows]
spark.createDataFrame(rows, schema).write.mode("overwrite").parquet(f"{VOLUME_PATH}/products/")
print(f"Generated {NUM_PRODUCTS} products")

In [ ]:
# --- Generate Orders ---
end_date = datetime.now()
start_date = end_date - timedelta(days=DATE_RANGE_DAYS)

rows = []
for i in range(1, NUM_ORDERS + 1):
    order_date = start_date + timedelta(days=rng.randint(0, DATE_RANGE_DAYS))
    if order_date.weekday() >= 5 and rng.random() < 0.3:
        order_date += timedelta(days=2)
    status = rng.choices(ORDER_STATUSES, weights=STATUS_WEIGHTS, k=1)[0]
    rows.append((i, rng.randint(1, NUM_CUSTOMERS), order_date, status))

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("status", StringType(), False),
])
spark.createDataFrame(rows, schema).write.mode("overwrite").parquet(f"{VOLUME_PATH}/orders/")
print(f"Generated {NUM_ORDERS} orders")

In [ ]:
# --- Generate Order Items (Pareto product distribution) ---
popular_products = list(range(1, max(2, NUM_PRODUCTS // 5 + 1)))
other_products = list(range(NUM_PRODUCTS // 5 + 1, NUM_PRODUCTS + 1))

product_prices = {pid: round(max(5.0, min(math.exp(rng.gauss(3.5, 1.2)), 5000.0)), 2) for pid in range(1, NUM_PRODUCTS + 1)}

rows = []
item_id = 1
for order_id in range(1, NUM_ORDERS + 1):
    num_items = rng.choices([1, 2, 3, 4, 5], weights=[0.3, 0.35, 0.2, 0.1, 0.05], k=1)[0]
    for _ in range(num_items):
        product_id = rng.choice(popular_products) if (rng.random() < 0.8 and popular_products) else rng.choice(other_products or popular_products)
        quantity = rng.choices([1, 2, 3, 5, 10], weights=[0.4, 0.25, 0.15, 0.12, 0.08], k=1)[0]
        effective_price = round(product_prices.get(product_id, 50.0) * rng.uniform(0.85, 1.15), 2)
        rows.append((item_id, order_id, product_id, quantity, effective_price, round(effective_price * quantity, 2)))
        item_id += 1

schema = StructType([
    StructField("item_id", LongType(), False),
    StructField("order_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("total_price", DoubleType(), False),
])
spark.createDataFrame(rows, schema).write.mode("overwrite").parquet(f"{VOLUME_PATH}/order_items/")
print(f"Generated {len(rows)} order items")
print("\nData generation complete!")